# B2-024 — Practice p11

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** mixture-parameter-regression

## Task

Implement the three-bump mixture function and its canonical ordering.

**Contract.**

* `mixture_function(x, weights, centers, widths)`: `x` is `(P,)`; `weights`, `centers`, `widths` are `(N, 3)`; returns `(N, P)` with entry $\sum_k w_{nk}\exp(-(x_p-c_{nk})^2/(2s_{nk}^2))$, in the input dtype, computed by broadcasting (no Python loops).
* `canonicalize(params)`: `params` is `(N, 9)` in the layout `(w1, c1, s1, w2, c2, s2, w3, c3, s3)`; returns a new `(N, 9)` tensor in which, for each row, the three `(w, c, s)` triples are reordered by increasing center (stable for equal centers), each triple kept intact. The input is not modified.

**Probes.** Load the unit loader with the supplied cell.

1. `mixture_function(X5, W, C, S)` equals `EXPECTED_F` with `ATOL = RTOL = 1e-10` (float64).
2. `canonicalize(RAW)` equals `CANON` exactly (`torch.equal`), and `canonicalize(CANON)` equals `CANON` (idempotence).
3. On `theta = capstone_data.train_rows("mixture")[1]` (shape `(800, 9)`, already canonical): `torch.equal(canonicalize(theta), theta)`; for the component permutation `(2, 0, 1)` applied to every row, `canonicalize` of the permuted tensor equals `theta` exactly; and the permuted and original parameters give the same `mixture_function` values on `capstone_data.MIXTURE_X` within `atol = rtol = 1e-6`.
4. On the validation rows `values, theta_v = capstone_data.val_rows("mixture")`: `mixture_function` of `theta_v` matches `capstone_data.mixture_values(capstone_data.MIXTURE_X, theta_v)` with `atol = rtol = 1e-6`, and the residual `values - mixture_function(...)` has a standard deviation between `0.015` and `0.025` (the generator's noise is `0.02`, plus rounding).
5. Wrong variant: a copy that sorts the triples by weight instead of center fails probe 2.

**Required answer-check assertions.** Probes 1–5, plus output shapes `(1, 5)` in probe 1 and `(800, 9)` in probe 3.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

X5 = torch.tensor([0.0, 0.25, 0.5, 0.75, 1.0], dtype=torch.float64)
W = torch.tensor([[1.0, 0.5, 1.2]], dtype=torch.float64)
C = torch.tensor([[0.7, 0.2, 0.45]], dtype=torch.float64)
S = torch.tensor([[0.05, 0.08, 0.06]], dtype=torch.float64)
EXPECTED_F = torch.tensor([[2.1968466812e-02, 4.1592788537e-01, 8.4875530921e-01, 6.0653513172e-01, 1.5229979745e-08]],
                          dtype=torch.float64)
RAW = torch.tensor([[1.0, 0.7, 0.05, 0.5, 0.2, 0.08, 1.2, 0.45, 0.06]], dtype=torch.float64)
CANON = torch.tensor([[0.5, 0.2, 0.08, 1.2, 0.45, 0.06, 1.0, 0.7, 0.05]], dtype=torch.float64)
ATOL = 1e-10
RTOL = 1e-10

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.